 # Dungeon'n'Dragons Characterizer #
 Convert your own personality into a real DnD character to enjoy the gAIme!<br>
 <i>Excersise based on Week 1 info</i>


In [1]:
# base imports, settings and initialization of variables 

import json
import os
import sys

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# add week1 directory to sys.path to get scraper module into imports
week1_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'week1'))
sys.path.append(week1_dir)
from scraper import fetch_website_contents, fetch_website_links


# load environment variables
load_dotenv(override=True)
MODEL = "gpt-5-mini"

# assign OPENAI API key from env and check if key exists and is valid
if not (openai_api_key := os.getenv('OPENAI_API_KEY')) or not openai_api_key.startswith("sk-proj-"):
    raise("No API key was found or is invalid (doesn't start sk-proj")
print("API key found and looks good so far!")

openai = OpenAI()



API key found and looks good so far!


Prepare knowledge base (as a list dictionaries) containing information about page (url, type and summary)

In [2]:
def get_knowledge_base_links(url_base, urlinker_user_prompt):
    '''
    Provide "universal" link selector with help of common system prompt specification - thematic usage depends on request in external user prompt
    '''
    all_urls = fetch_website_links(url_base)

    urlinker_system_prompt = """
    You are provided with a list of links found on a webpage dedicated to certain theme. Based on given intention you are able to decide which of the links 
    would be the most relevant as links necessary for the requested purpose. For example, when you get a company page and the intention is to create a brochure 
    about the company, then you shoudl involve pages like About, Vision and Mission, Annual report or Job opportunities. When you get webpages with soccer rules 
    and the intention is to create a training manual, then you should involve pages like Basic rules, Ball typology, Selecting best football boots. 
    You should respond in JSON as in this example:
    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about_page", "summary": "short summary of information found on the page"},
            {"type": "game rules page", "url": "https://another.full.url/game_rules", "summary": "short summary of information found on the page"}
        ]
    }
    """

    messages = [
        {"role": "system", "content": urlinker_system_prompt},
        {"role": "user", "content": urlinker_user_prompt + f"\nChoose relevant links from following list: {all_urls}."}
    ]

    print("Consideration of links relevancy started...")
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    knowledge_base_links = json.loads(response.choices[0].message.content)

    print(f"For website {url_base} {len(knowledge_base_links['links'])} of {len(all_urls)} considered as relevant")
    return knowledge_base_links 


Get all relevant links from Dungeon and Dragons rule-site (only links containing information necessary for character creation should be involved).<br>User prompt shoudl be specific.

In [3]:
dnd_url_base = "https://www.dndbeyond.com/sources/dnd/basic-rules-2014"

urlinker_user_prompt = f"""
You are provided with a webpage links dedicated to a famous table-top role playing game Dungeon and Dragons. The intention is to create a character, 
so the only relevant pages are those dedicated to creation of a game character like choosing a race, personal characteristics, tools or abilities and similar. 
"""

dnd_kb = get_knowledge_base_links(dnd_url_base, urlinker_user_prompt)


Consideration of links relevancy started...
For website https://www.dndbeyond.com/sources/dnd/basic-rules-2014 25 of 195 considered as relevant


Compose DnD character - source information is taken from predefined knowledge base and a personal description of a human player 

In [ ]:

player_websites = [
    fetch_website_contents("https://edwarddonner.com"),
    fetch_website_contents("https://en.wikipedia.org/wiki/Elizabeth_II"),
    fetch_website_contents("https://en.wikipedia.org/wiki/Arnold_Schwarzenegger")
]
personal_summaries = [
    openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Act as helpful snearky assistent."},
            {"role": "user", "content": f"Provide a personal summary from website: {website}"}
        ]
    ) for website in player_websites
]

system_prompt = f"""
Act as a character designer for Dungeon and Dragons player with access to DnD website links summarized in following JSON:\n{json.dumps(dnd_kb)}\n  
Your task is to compose suitable DnD character based on a real characteristics of the human player. For the character's skills do not try to maximize 
the abilities intentionaly but reflect personal information. Do not use Human race for generated character but choose any other suitable race instead.
After the creation of the character write a short explanation why you decided right so"""

for personal_summary in personal_summaries:
    
    user_prompt = f"""Prepare suitable DnD character for a human player described by his or her personal information: {personal_summary}. 
    If possible then use the same style as in previously generated characters."""    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    dnd_character_stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in dnd_character_stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)
    
    # extend user_prompt with "chat history" to keep initial style for all characters
    if "assistant" not in [item['role'] for item in messages]:
        messages.append({"role": "assistant", "content": response})


Ed Donner — the LLM-obsessed, teacher-by-day, amateur electronic musician-by-night — as a D&D character.

Short character sheet (level 1)
- Name: Ed Donner
- Race: Rock Gnome (small, tinkering-minded, +2 Int, +1 Con)
- Class: Wizard (scholar / tinkerer vibe)
- Background: Sage (researcher & educator)
- Alignment: Neutral Good
- HP: 8 (1d6 + CON mod)
- Proficiency bonus: +2

Ability scores (standard array; racial bonuses applied)
- Strength 8 (–1)
- Dexterity 14 (+2)
- Constitution 13 (+1) -> 14 (+2) after Rock Gnome +1
- Intelligence 15 (+2) -> 17 (+3) after Gnome +2
- Wisdom 10 (+0)
- Charisma 12 (+1)

Skills (chosen to reflect real-life strengths, not to min-max)
- Arcana (Int) — research & technical knowledge (LLMs/algorithms)
- Investigation (Int) — debugging, problem solving
- History (Int) — curriculum-building, long-form knowledge
- Insight (Wis) — reading people (lectures, community interaction)

Tool proficiencies & languages
- Tinkerer’s tools (Rock Gnome tinker trait) — electronics & gadgets
- Languages: Common, Gnomish, plus two scholarly tongues (e.g., Elvish, Draconic)

Starting equipment (typical wizard + flavor)
- Spellbook
- Quarterstaff (arcane focus) + dagger
- Component pouch / scholar’s pack
- Tinker’s tools and a few “kits” of electronic bits (trinkets for flavor)

Cantrips (signature toolkit)
- Mage Hand — remote manipulation of hardware / live demos
- Prestidigitation — small experiments, stage tricks for lectures
- Message — quick, private communications (chatty LLM analog)

1st-level spells (examples to match a tech/scholar theme)
- Detect Magic — investigative tool (spot hidden layers)
- Identify — analyze devices, spells, systems
- Comprehend Languages — instant translation for global students
- Shield — defensive backup when things go sideways
- Magic Missile — reliable, predictable output (like a well-tested system)

Brief backstory (snack-sized, high-energy)
Ed Donner, Codewright of the Nebula, used to run a startup that got folded into something bigger; now he runs experiments in arcane cognition and teaches hundreds of thousands across the globe. He keeps a cluttered tower-workshop of synths and circuit boards, lectures colleagues with the blunt charm of a Hacker News thread, and writes meticulous notes in his spellbook about agentic systems. He’s happiest debugging a spell loop at 3 a.m. while a half-finished synth patch drones in the background.

Why I chose these options (short explanation)
- Race: Rock Gnome — Gnomes naturally evoke tinkerers, inventors, and bright, curious minds. That matches Ed’s engineering, electronics, and maker leanings far better than Human (as you requested “not Human”).
- Class: Wizard — Ed is a researcher/teacher and an LLM/AI builder: a wizard’s focus on study, books, and structured, replicable knowledge mirrors an engineer’s life of models, tests, and curricula. Wizard spells map nicely to analysis/diagnostics (Identify, Detect Magic) and communication (Comprehend Languages, Message).
- Background: Sage — obvious fit for an educator and researcher; gives the “I know where to look” narrative and supports the teaching/author persona.
- Abilities & skills: I deliberately didn’t max everything. Int is clearly the highest stat (reflecting deep technical skill), Con and Dex are respectable to represent staying up late and fiddling safely with gear, and Charisma is moderate (Ed teaches and presents widely but isn’t a natural-born performer). Skill choices reflect his real-world strengths: Arcana/Investigation/History for technical knowledge and pedagogy, Insight for reading audiences and collaborators.
- Tools and spells: Tinker’s tools, Mage Hand, and Prestidigitation highlight the electronics/music hobby and the practical, hands-on side of his work; spells like Identify and Comprehend Languages echo analysis, debugging, and teaching hundreds of thousands of students worldwide.

If you want: I can:
- Tune this for a different class (Bard for a music-forward Ed, Artificer if your table allows it),
- Make a level-up plan (which spells, feats, or subclass match Ed’s career path),
- Or write a short one-paragraph intro Ed could read at the table in character.

Regina Alexandra, “The Constant”
Short, snappy D&D character inspired by the player info you gave.

Core
- Race: Half‑elf (long‑lived, composed, naturally dignified — not Human as requested)
- Class (level 1): Paladin — Oath of Duty / Oath of Devotion flavor (duty, ceremony, protecting the realm)
- Background: Former ATS Mechanic (custom: Soldier variant + royal household duties) — reflects wartime service and lifelong ceremonial role
- Alignment: Lawful Neutral (upholds institution and duty; stays publicly neutral)

Ability scores (final)
- STR 10 (+0)
- DEX 10 (+0)
- CON 13 (+1)  — sturdy, long‑lived
- INT 12 (+1)
- WIS 15 (+2)  — good judgment, experience
- CHA 16 (+3)  — regal presence, public figure
(half‑elf racial: +2 CHA, +1 CON, +1 WIS already applied)

Proficiencies
- Saving throws: Wisdom, Charisma
- Skills (chosen to match the person, not to min‑max): Insight, Persuasion, History, Animal Handling
- Background/tool proficiencies (custom ATS mechanic): Land Vehicles (vehicle) proficiency, Smith’s or Tinker’s tools (for vehicle/engine maintenance)
- Armor/weapons: All armor, shields, simple and martial weapons

Key features & equipment
- Lay on Hands (paladin 1): caretaker, ceremonial healer-like presence
- Divine Sense: sense of right/wrong in solemn situations (ceremonial authority)
- Starting gear: chain mail, shield, longsword, holy symbol/emblem of the realm, riding tack, a small corgi trinket (personal token), toolkit for vehicle repair

Personality snapshot (3 lines)
- Duty first: steady, unflappable, keeps to tradition.
- Private mechanic curiosity: knows how to get into a vehicle and fix it if needed.
- Deep affection for animals and pageantry—horses and small dogs get special treatment.

Roleplaying hooks
- Meets leaders and negotiates with quiet authority (Persuasion + Insight).
- Keeps public neutrality, but privately protects the realm’s traditions and people.
- Keeps a tucked toolkit and will discreetly roll up sleeves to help in a crisis.

Why these choices
- Half‑elf preserves a non‑Human restriction while giving natural charismatic presence and versatility appropriate for a long‑reigning, dignified public figure.
- Paladin fits the sense of duty, ceremony, and moral steadiness that defined the player’s profile; Lay on Hands and Divine Sense map to caretaker and moral‑authority roles.
- The custom Soldier/ATS mechanic background honors wartime service — driving and maintaining vehicles — so tool proficiencies and the “former mechanic” flavor are included rather than shoehorning pure Noble traits.
- Skills were chosen to reflect real‑world strengths (Persuasion for public duties; Insight for meeting prime ministers; Animal Handling for horses/corgis; History for long memory and institutional knowledge) rather than to maximize combat effectiveness.

If you want, I can:
- Turn this into a level 3 paladin with an explicit oath subclass (Oath of the Crown / Devotion flavor),
- Swap class to Bard (College of Lore) or Cleric (Knowledge) if you prefer diplomacy/ceremony to martial authority,
- Or make a shorter printable character sheet for tabletop use.

Alright — using the same concise, punchy style as your earlier example, here’s a D&D character built from the supplied player portrait (the Schwarzenegger-style summary). I avoided Human and I reflected real-life strengths and flaws in the skills rather than min-maxing.

Character snapshot
- Name: Arnod Stahl (player alias; use any name you prefer)
- Race: Half-Orc (fits a towering, competitive, tough background without using Human)
- Class & subclass: Fighter (Champion) — physical powerhouse and performer; Champion reflects raw athletic prowess and iconic “bigger-than-life” presence
- Level: 1 (ready to advance; scales smoothly with Fighter progression)
- Background: Entertainer (bodybuilding champion → movie star / public performer)
- Alignment: Neutral Good (pragmatic, likes results; public service motivation but sometimes blunt)

Ability scores (point-buy style; racial bonuses applied)
- Strength 17 (+3) — built for winning competitions, lifting, and big action feats
- Constitution 15 (+2) — durable and injury-hardened
- Dexterity 10 (+0) — serviceable but not nimble
- Intelligence 10 (+0) — average book-smarts
- Wisdom 8 (−1) — not always perceptive or tactful in subtle social situations
- Charisma 13 (+1) — stage presence and magnetism, enough for celebrity and politics but not a charisma specialist

Proficiencies and saving throws
- Armor/Weapons: all armor, shields, simple & martial weapons (Fighter)
- Saving throws: Strength, Constitution
- Skills (reflecting biography — not optimized, just true-to-life):
  - Athletics (Fighter) — expert at feats of strength
  - Intimidation (Half-Orc flavor / background) — imposing presence from years of competition and action roles
  - Performance (Entertainer) — stage and cinematic presence; can command crowds
  - Persuasion (background choice) — some success convincing crowds & voters, but not an absolute social genius
- Tools: Disguise kit and one instrument (Entertainer) — for stagecraft and role-play
- Languages: Common, Orc

Race features (Half-Orc highlights)
- Darkvision (useful in low light)
- Menacing: proficiency in Intimidation (if you want to swap)
- Relentless Endurance: drop to 1 HP instead of 0 once per long rest (suits comeback-from-defeat narrative)
- Savage Attacks: extra damage on critical hits (fits cinematic “one strike” moments)

Class features (Fighter 1)
- Fighting style: Great Weapon Fighting (fits the big-muscle movie trope; take a greataxe/ greatsword)
- Second Wind: brief self-heal (bulwark of resilience)

Suggested equipment (starting)
- Greatsword or greataxe (big, showy weapon)
- Chain mail or half-plate (if starting gear options allow; otherwise chain mail)
- Explorer’s pack or entertainer’s pack (Entertainer grants a costume and travel-friendly kit)
- Disguise kit, performer’s trinkets (posters, trophies)

Personality & roleplaying hooks (drawn from the profile)
- Personality trait: “I know how to win and I expect to do so; I show confidence whether lifting or walking onto a stage.”
- Ideal: Ambition — “Hard work and perseverance move mountains.”
- Bond: “I owe my success to my first coach/community back home; I’ll protect and promote them.”
- Flaw: “I sometimes boast or lean on blunt tactics; past missteps can make others distrust my motives.”

Backstory sketch
Born in a rough alpine village, Arnod trained to be the best strongman. He rose to fame through contests, then parlayed that fame into a career entertaining crowds and starring in action spectacles. Later he stepped into leadership to reshape his adopted city/region — strong on big gestures and environmental projects, but sometimes awkward or politically unpopular in private controversies. He’s a living legend, still driven to outperform and inspire.

Why I made these choices
- Race: Half-Orc captures the imposing physicality and resilience of the real-life portrait (big, durable, and occasionally intimidating), while avoiding Human per your constraint.
- Class/subclass: Fighter (Champion) mirrors the athlete-to-action-star arc: raw physical dominance, straightforward combat style, and the Champion’s emphasis on crits and peak performance matches a “larger-than-life” competitor-actor. It’s mechanically simple and thematically on-point.
- Background: Entertainer reflects bodybuilding competitions and a film career; it also gives the social tools to be a public figure without making the character a charisma specialist.
- Skills: I gave Athletics, Performance, Persuasion, and Intimidation because they mirror the player’s bodybuilding, acting, political campaigning, and imposing public persona. I didn’t artificially pump CHA or skill picks beyond what’s realistic for the portrait — the character can charm or lead a crowd, but isn’t an optimized social manipulator.
- Flaws & nuance: The bio includes controversies and political awkwardness; I folded that into lower Wisdom and a social flaw so the PC has roleplaying depth and complications to explore.

If you want: level this to 5 for Extra Attack and a more cinematic combat presence; or change subclass to Battle Master if you prefer a tactical/political strategist rather than raw champion. I can also produce a one-page printable character sheet for your chosen level. Which would you like next?